# 02. V1 도구 개발 (Tools V1)

> Part 02에서 `@tool`, `bind_tools`, `ToolNode`로 도구 루프의 기본을 배웠다면, 여기서는 V1 도구가 **런타임 정보에 접근하고 상태를 업데이트하는 방법**을 배웁니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. Part 02의 `@tool` 기초를 V1 에이전트 도구 정의에 적용할 수 있어요
2. Pydantic `args_schema`로 타입 검증과 입력 제한을 구현할 수 있어요
3. `ToolRuntime`을 통해 에이전트 상태, 컨텍스트, 메모리 스토어에 접근할 수 있어요
4. `Command`와 `RemoveMessage`로 도구 내에서 상태를 동적으로 업데이트할 수 있어요
5. `runtime.stream_writer`로 도구 실행 진행 상황을 실시간 스트리밍할 수 있어요

## 사전 지식

- `02_LangGraph_Basics/06-Tools-Integration.ipynb`: `@tool`, `bind_tools`, `ToolNode`, `tools_condition`
- `01-Create-Agent.ipynb`: `create_agent`, `AgentState`, 에이전트 루프
- Pydantic BaseModel 기본 사용법


## V1 도구에서 새로 보는 것

도구가 “LLM이 호출할 수 있는 함수”라는 기본 개념은 `02_LangGraph_Basics/06-Tools-Integration.ipynb`에서 이미 배웠어요. 이 노트북의 초점은 기본 도구 루프가 아니라, V1 도구가 실행 중에 접근할 수 있는 **런타임 표면**입니다.

```
도구 = 함수 + 이름 + 설명 + 입력 스키마
V1 도구 확장 = ToolRuntime + Command + stream_writer + store/context 접근
```

### V1 도구의 주요 구성 요소

| 구성 요소 | Part 02와의 관계 | 이 노트북의 초점 |
|----------|------------------|------------------|
| `@tool` 데코레이터 | 기본 사용법 복습 | 이름/설명/스키마 품질 점검 |
| `args_schema` | Part 02보다 상세 | Pydantic 검증과 `Literal` 제한 |
| `ToolRuntime` | 새 초점 | state, context, store, stream_writer 접근 |
| `Command` 반환 | 새 초점 | 도구 실행 중 상태 업데이트 |
| `stream_writer` | 새 초점 | 장시간 도구 진행 상황 전송 |

> 🔑 **핵심 개념**: LLM은 여전히 도구의 이름·설명·스키마를 보고 호출을 결정해요. 하지만 V1 도구는 `ToolRuntime`을 통해 LLM에게 노출하지 않는 실행 컨텍스트까지 사용할 수 있습니다.


## 환경 설정

In [39]:
# 환경 변수 로드 (.env 파일에서 OPENAI_API_KEY 등을 읽어와요)
from dotenv import load_dotenv
load_dotenv()

# LangSmith 추적 설정 (선택 사항)
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Tools"

## 1. @tool 데코레이터로 기본 도구 만들기

`@tool`의 기본 동작은 Part 02에서 다뤘으므로 여기서는 빠르게 복습합니다. Python 함수에 데코레이터를 붙이면 이름, docstring, 타입 힌트가 도구 스키마로 추출돼요.

**자동 추출 항목:**
- **이름**: 함수 이름 (`search_database` → `search_database`)
- **설명**: 함수의 docstring
- **입력 스키마**: 타입 힌트에서 자동 생성

> 💡 **핵심 정리**: 이 섹션은 이후 `args_schema`, `ToolRuntime`, `Command`를 이해하기 위한 짧은 베이스라인입니다. 도구 루프 전체 구조는 `02_LangGraph_Basics/06-Tools-Integration.ipynb`를 참조하세요.


In [40]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    # 실제로는 DB 쿼리를 수행하지만, 여기서는 시뮬레이션해요
    return f"Found {limit} results for '{query}'"


# 도구의 메타데이터를 확인해봐요
print(f"도구 이름: {search_database.name}")
print(f"도구 설명: {search_database.description}")
print(f"입력 스키마: {search_database.args}")


도구 이름: search_database
도구 설명: Search the customer database for records matching the query.

Args:
    query: Search terms to look for
    limit: Maximum number of results to return
입력 스키마: {'query': {'title': 'Query', 'type': 'string'}, 'limit': {'default': 10, 'title': 'Limit', 'type': 'integer'}}


### 1-1. 커스텀 이름과 설명 설정

함수 이름 대신 더 설명적인 도구 이름을 쓰고 싶을 때, 또는 docstring과 별도로 모델에게 다른 설명을 전달하고 싶을 때 사용해요.

> 💡 **실무 팁**: 도구 이름은 `snake_case`로 작성하고, 동사로 시작하는 것이 좋아요 (`search_`, `get_`, `create_`, `update_`). 모델이 도구의 목적을 이름만으로도 유추할 수 있어요.

In [41]:
@tool("web_search")
def search(query: str) -> str:
    """Search the web for information"""

    return f"Result for : {query}"

# description 파라미터로 모델에게 전달할 설명을 별도 지정해요
@tool(
    "calculator",
    description="Performs arithmetic calculations. Use this for any math problems including addition, subtraction, multiplication and division.",
)
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    try:
        # 실제 사용 시 안전한 파서를 사용하세요
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {e}"

print(search.name)
print(calc.description)

web_search
Performs arithmetic calculations. Use this for any math problems including addition, subtraction, multiplication and division.


## 2. Pydantic으로 입력 스키마 정의하기

파라미터가 복잡하거나 입력값에 제한이 필요할 때는 Pydantic 모델로 `args_schema`를 명시해요.

**Pydantic 스키마의 장점:**
- `Literal["celsius", "fahrenheit"]` 같은 허용값 제한
- `Field(description=...)` 으로 파라미터별 상세 설명
- 자동 타입 검증 및 변환

> 🔑 **핵심 개념**: `Field(description=...)` 에 작성한 내용이 모델에게 전달돼요. "어떤 값을 넣어야 하는지" 모델이 더 잘 이해할 수 있도록 구체적으로 작성하세요.

In [42]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """날씨 조회 도구의 입력 스키마"""
    location: str = Field(description="City name (e.g Seoul, Tokyo) or cordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit: 'celsius' or 'fahrenheit'"
    )
    include_forecast: bool = Field(
        default=False,
        description="Set True to include 5 day weather forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str="celsius", include_forecast: bool=False):
    """
    Get current weather and optional 5 days forecast for a location
    """
    # 온도 단위에 따른 기본 온도 시뮬레이션
    temp = 22 if units == "celsius" else 72
    unit_label = "°C" if units == "celsius" else "°F"
    result = f"{location} 현재 온도: {temp}{unit_label}"

    if include_forecast:
        result += "\n5일 예보: 맑음 > 구름 > 비 > 맑음 > 맑음"
    return result

print(get_weather.invoke({"location": "Seoul", "units": "celsius", "include_forecast": True}))


Seoul 현재 온도: 22°C
5일 예보: 맑음 > 구름 > 비 > 맑음 > 맑음


In [43]:
try:
    print(get_weather.invoke({"location": "Seoul", "units": "kelvin", "include_forecast": True}))
except Exception as e:
    print(f"검증 오류 발생: {e}")

검증 오류 발생: 1 validation error for WeatherInput
units
  Input should be 'celsius' or 'fahrenheit' [type=literal_error, input_value='kelvin', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


> 💡 **핵심 정리**: `Literal` 타입은 단순한 검증 이상의 역할을 해요. 모델이 스키마를 읽을 때 허용값 목록을 보고 "celsius 아니면 fahrenheit"라고 학습하기 때문에, 잘못된 값을 입력할 가능성이 처음부터 줄어들어요.

## 3. ToolRuntime - 런타임 정보 접근

`ToolRuntime`은 도구가 실행될 때 주입되는 특별한 매개변수예요. 에이전트의 상태, 사용자 컨텍스트, 장기 메모리 스토어 등에 접근할 수 있어요.

**중요한 특성:**
- `runtime: ToolRuntime`을 도구 시그니처에 추가하면 모델에게는 **보이지 않아요**
- 실행 시 LangGraph 프레임워크가 자동으로 주입해요
- 단일 객체로 여러 컨텍스트 소스에 접근 가능

```mermaid
flowchart LR
    TR[ToolRuntime] --> S[.state<br/>현재 그래프 상태]
    TR --> C[.context<br/>불변 사용자 컨텍스트]
    TR --> ST[.store<br/>영구 장기 메모리]
    TR --> SW[.stream_writer<br/>실시간 스트리밍]
    TR --> ID[.tool_call_id<br/>현재 도구 호출 ID]

    classDef runtime fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef prop fill:#cce5ff,stroke:#007bff,color:#004085
    class TR runtime
    class S,C,ST,SW,ID prop
```

### ToolRuntime 속성별 용도 비교

| 속성 | 데이터 성격 | 지속성 | 대표 사용 사례 |
|------|-----------|--------|-------------|
| `.state` | 대화 중 변하는 동적 데이터 | 세션 내 | 메시지 이력, 카운터 |
| `.context` | 요청 시 주입되는 불변 데이터 | 단일 호출 | user_id, 권한, 설정 |
| `.store` | 세션을 넘어 영속하는 데이터 | 영구 | 사용자 선호도, 검색 기록 |
| `.stream_writer` | 실시간 이벤트 전송 | 즉시 | 진행률, 중간 결과 |
| `.tool_call_id` | 현재 호출 식별자 | 단일 호출 | Command 반환 시 필수 |

> 💡 **실무 팁**: `runtime.state`는 읽기 전용으로 사용하고, 상태를 바꿔야 할 때는 `Command`를 반환하세요. 직접 state를 변경하면 예측 불가능한 동작이 생길 수 있어요.

### 3-1. runtime.state - 현재 대화 상태 접근

In [44]:
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# runtime.state로 현재 대화 히스토리에 접근해요
@tool
def summarize_conversation(runtime: ToolRuntime) -> str:
    """현재 대화의 메시지 통계를 반환해요."""
    messages = runtime.state.get("messages", [])

    # 메시지 타입별 카운트
    human_count = sum(1 for m in messages if m.__class__.__name__ == "HumanMessage")
    ai_count = sum(1 for m in messages if m.__class__.__name__ == "AIMessage")
    tool_count = sum(1 for m in messages if m.__class__.__name__ == "ToolMessage")

    return (
        f"대화 통계 - "
        f"사용자: {human_count}개, "
        f"AI 응답: {ai_count}개, "
        f"도구 결과: {tool_count}개"
    )

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=[summarize_conversation],
    system_prompt="너는 도움을 주는 어시스턴트야."
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "지금까지 대화 통계를 알려줘."}]}
)

result

{'messages': [HumanMessage(content='지금까지 대화 통계를 알려줘.', additional_kwargs={}, response_metadata={}, id='59ccb6af-5b84-4afb-a91d-2960ad3e9fc7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 67, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b7b4c219e8', 'id': 'chatcmpl-DloYRUliG3Timtr2RZ7tQIJxIYMQP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8170-e28b-7bb2-a73e-d596a2636a6b-0', tool_calls=[{'name': 'summarize_conversation', 'args': {}, 'id': 'call_DtdZQIuv2bOkAXBMaCMLT5We', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 67, 'output_tokens': 13, 'to

In [45]:
from IPython.display import Image, display

agent

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

### 3-2. runtime.context - 사용자 컨텍스트 활용

In [46]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 사용자 데이터베이스 시뮬레이션
USER_DATABASE = {
    "user123": {
        "name": "김철수",
        "account_type": "Premium",
        "balance": 5_000_000,
        "email": "chulsoo@example.com",
    },
    "user456": {
        "name": "이영희",
        "account_type": "Standard",
        "balance": 1_200_000,
        "email": "younghee@example.com",
    },
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """현재 로그인한 사용자의 계좌 정보를 조회한다."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"계좌 소유자: {user['name']}\n"
            f"계좌 유형: {user['account_type']}\n"
            f"잔액: {user['balance']:,}원"
        )
    else:
        return "사용자가 없습니다."

# context_schema 파라미터로 컨텍스트 스키마를 에이전트에 등록해요
model = init_chat_model("openai:gpt-4o-mini")
agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,  # 클래스 자체를 전달 (인스턴스가 아님)
    system_prompt="You are a banking assistant. Respond in Korean.",
)

user_a_context = UserContext(user_id="user123")
result = agent.invoke(
    {"messages": [{"role": "user","content": "내 계좌 잔액이 얼마야?"}]},
    context=user_a_context
)

result

{'messages': [HumanMessage(content='내 계좌 잔액이 얼마야?', additional_kwargs={}, response_metadata={}, id='f5d95413-aa04-481b-bcc5-15ca62b4a3cb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 63, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_90d8d1ae6b', 'id': 'chatcmpl-DloYbSrKcaYhau7MxZ379k0H3F1Tx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8171-0876-7ed2-8e94-cbcc526cd3af-0', tool_calls=[{'name': 'get_account_info', 'args': {}, 'id': 'call_NurX30cDdGeR0SlltwKhosU0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 11, 'total_token

> 💡 **핵심 정리**: `context`는 매 요청마다 달라지는 정보를 전달할 때 써요. 예를 들어 웹 앱에서 로그인한 사용자의 ID나 권한 레벨을 context로 넘기면, 도구가 요청자에 맞는 데이터만 반환할 수 있어요.

> 🔑 **핵심 개념**: `state`는 대화 중 바뀌는 동적 데이터 (`messages`, 카운터 등)이고, `context`는 요청이 들어올 때 한 번 설정되는 불변 데이터 (`user_id`, 세션 설정 등)예요.

### 3-3. runtime.store - 장기 메모리 스토어

In [47]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """저장된 사용자 정보를 조회한다"""
    item = runtime.store.get(("users",), user_id)

    return str(item.value) if item else f"{user_id} 사용자 정보 입력"

@tool
def save_user_info(user_id, user_info: dict[str, Any], runtime: ToolRuntime):
    """사용자 정보를 장기 메모리 스토어에 저장한다"""
    runtime.store.put(("users",), user_id, user_info)

store = InMemoryStore()
model = init_chat_model("openai:gpt-4o-mini")
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store,
    system_prompt="너는 도움을 주는 어시스턴트야"
)

res1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "사용자 정보를 지정해. 사용자 아이디는 u001, 이름은 곽두팔, 나이는 서른살, 직업은 프로레슬러야."}
        ]
    }
)

print(res1["messages"][-1].content)


사용자 정보가 저장되었습니다. 이제 필요하신 다른 정보나 도움이 필요하시면 말씀해 주세요!


In [48]:
res2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "u001 사용자 정보 알려줘."}
        ]
    }
)

print(res2["messages"][-1].content)

u001 사용자 정보입니다:

- 이름: 곽두팔
- 나이: 30
- 직업: 프로레슬러


> 💡 **실무 팁**: `InMemoryStore`는 프로세스가 종료되면 데이터가 사라져요. 프로덕션에서는 `PostgresStore` 또는 `RedisStore`를 사용해서 영구 저장하세요. Part 7에서 장기 메모리를 더 자세히 다뤄요.

## 4. Command로 도구 내에서 상태 업데이트하기

일반 도구는 문자열을 반환해서 `ToolMessage`로 변환되지만, `Command`를 반환하면 **도구 실행 중에 에이전트 상태를 직접 업데이트**할 수 있어요.

```python
return Command(
    update={
        "user_name": new_name,      # 커스텀 상태 필드 업데이트
        "messages": [ToolMessage(   # 메시지 히스토리에 추가
            content="완료",
            tool_call_id=runtime.tool_call_id,  # 필수!
        )]
    }
)
```

> ⚠️ **자주 하는 실수**: `Command`로 상태를 업데이트할 때 `messages`에 반드시 `ToolMessage`를 포함해야 해요. `tool_call_id`도 `runtime.tool_call_id`에서 정확히 가져와야 에이전트가 어떤 도구 호출에 대한 응답인지 추적할 수 있어요.

In [49]:
from typing_extensions import TypedDict
from anyio.lowlevel import checkpoint
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent, AgentState
from langchain.chat_models import init_chat_model
from langchain.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

class CustomState(AgentState):
    user_name: str

@tool
def update_user_name(new_name: str, runtime: ToolRuntime) -> Command:
    """Update the user's name in the conversation state."""
    return Command(
        update = {
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f"이름을  '{new_name}'으로 저장했어요",
                    tool_call_id=runtime.tool_call_id
                )
            ]
        }
    )

model = init_chat_model("openai:gpt-4o-mini")
agent = create_agent(
    model,
    tools=[update_user_name],
    system_prompt="You are a helpful assistant. Respond in Korean",
    state_schema=CustomState,
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "user01-thread"}}
result1 = agent.invoke(
    {"messages":[{"role": "user", "content": "내 이름은 두팔이야"}]},
    config=config
)

print(result1["messages"][-1].content)

state = agent.get_state(config).values

state


이름을 '두팔'로 성공적으로 저장했어요! 어떤 도움이 필요하신가요?


{'messages': [HumanMessage(content='내 이름은 두팔이야', additional_kwargs={}, response_metadata={}, id='dd8bdc16-c611-46af-8e24-2c0f6ae1f93a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 65, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_53ee395669', 'id': 'chatcmpl-DloYjfPWRLvnsDJAZpHmYDyE2ARbj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8171-267c-7c91-a9af-078a904f0ad0-0', tool_calls=[{'name': 'update_user_name', 'args': {'new_name': '두팔'}, 'id': 'call_bSTKlGAz6rMP0CiENS85l93s', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 17,

In [50]:
# 이름 변경 테스트
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "아 사실 내 이름은 지수야"}]},
    config=config,
)
# === 이름 변경 ===
print(result2["messages"][-1].content)

# 상태 다시 확인
state = agent.get_state(config).values
print(f"\n업데이트된 이름: '{state.get('user_name', '없음')}'")

이름을 '지수'로 변경했어요! 무엇을 도와드릴까요?

업데이트된 이름: '지수'


### 4-1. RemoveMessage로 메시지 히스토리 관리

`RemoveMessage`를 사용하면 대화 히스토리에서 특정 메시지를 삭제할 수 있어요. `Command`와 함께 사용해서 도구 내에서 히스토리를 정리할 수 있어요.

> 🔑 **핵심 개념**: 삭제할 메시지를 `RemoveMessage(id=msg.id)`로 지정해요. 단, 현재 도구 호출과 연결된 `AIMessage`(tool_calls 포함)는 삭제하면 안 돼요 - 삭제하면 응답 쌍이 깨져요.

In [51]:
from langchain.messages import AIMessage, RemoveMessage
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime

@tool
def clear_messages(runtime) -> Command:
    """Clear all conversation messages except the currnet tool call."""
    messages = runtime.state.get("messages", [])
    tool_call_id = runtime.tool_call_id

    to_remove = []

    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            # 현재 tool_call_id와 매칭되는 AIMessage는 유지
            if not any(call.get("id") == tool_call_id for call in m.tool_calls):
                to_remove.append(m)
        else:
            to_remove.append(m)

    removals = [RemoveMessage(id=m.id) for m in to_remove]

    return Command(
        update = {
            "messages": removals + [ToolMessage(content=f"{len(removals)} 개 메시지를 삭제했어요.", tool_call_id=tool_call_id)]
        }
    )

# 여러 도구를 함께 사용하는 에이전트
model = init_chat_model("openai:gpt-4o-mini")
agent_with_clear = create_agent(
    model,
    tools=[update_user_name, clear_messages],
    system_prompt="You are a helpful assistant. Respond in Korean.",
    state_schema=CustomState,
    checkpointer=InMemorySaver(),
)

config2 = {"configurable": {"thread_id": "session-002"}}

# 대화 쌓기
agent_with_clear.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 하늘이야"}]},
    config=config2,
)
agent_with_clear.invoke(
    {"messages": [{"role": "user", "content": "안녕! 오늘 날씨 어때?"}]},
    config=config2,
)

before_count = len(agent_with_clear.get_state(config2).values["messages"])
print(f"삭제 전 메시지 수: {before_count}개")

# 메시지 삭제
result = agent_with_clear.invoke(
    {"messages": [{"role": "user", "content": "대화 기록 전부 지워줘"}]},
    config=config2,
)
print(result["messages"][-1].content)

after_count = len(agent_with_clear.get_state(config2).values["messages"])
print(f"삭제 후 메시지 수: {after_count}개")


삭제 전 메시지 수: 6개
안녕하세요! 어떻게 도와드릴까요?
삭제 후 메시지 수: 3개


## 5. runtime.stream_writer로 진행 상황 스트리밍

시간이 오래 걸리는 작업을 수행할 때 사용자에게 진행 상황을 실시간으로 알려줄 수 있어요. `runtime.stream_writer`를 호출하면 해당 내용이 즉시 스트림으로 전송돼요.

> 💡 **실무 팁**: `stream_mode="custom"` 으로 그래프를 실행해야 `stream_writer`로 보낸 메시지를 받을 수 있어요. 일반 `stream_mode="updates"` 에서는 커스텀 스트림이 전달되지 않아요.

In [52]:
import time
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

@tool
def analyze_data(data_size: int, runtime: ToolRuntime) -> str:
    """데이터를 분석하고 요약한 내용을 반환"""

    writer = runtime.stream_writer

    writer({"step": 1, "message": f"데이터 {data_size}건 로딩중....."})

    writer({"step": 2, "message": "데이터 정제 중..."})
    time.sleep(0.1)

    writer({"step": 3, "message": "통계 계산 중..."})
    time.sleep(0.1)

    writer({"step": 4, "message": "분석 완료!"})

    return f"분석 결과: {data_size}건 데이터, 평균 및 표준편차"

model = init_chat_model("openai:gpt-4o-mini")
agent = create_agent(
    model,
    tools=[analyze_data],
    system_prompt="You are a data analysis assistant. Respond in Korean.",
)

inputs = {"messages": [{"role": "user", "content": "1000건 데이터를 분석해줘"}]}

for chunk in agent.stream(inputs, stream_mode="custom"):
    print(f"[진행] {chunk}")


[진행] {'step': 1, 'message': '데이터 1000건 로딩중.....'}
[진행] {'step': 2, 'message': '데이터 정제 중...'}
[진행] {'step': 3, 'message': '통계 계산 중...'}
[진행] {'step': 4, 'message': '분석 완료!'}


> 💡 **핵심 정리**: `stream_writer`는 UI 구현 시 핵심이에요. 웹 애플리케이션에서 진행 상황 바를 업데이트하거나, 단계별 로그를 사용자에게 보여줄 때 활용해요. 단계가 많고 시간이 오래 걸리는 RAG 파이프라인, 웹 스크래핑 등에서 특히 유용해요.

## 6. 종합 예제 - 개인화된 추천 에이전트

지금까지 배운 모든 내용을 합쳐서 실제 프로덕션에 가까운 에이전트를 만들어봐요.

- `@tool` + `args_schema`: 입력 검증이 있는 도구
- `runtime.context`: 사용자 ID 기반 개인화
- `runtime.store`: 사용자 선호도 영구 저장
- `Command`: 상태 업데이트

In [53]:
from typing import Annotated, List, Literal
from dataclasses import dataclass
from pydantic import BaseModel, Field
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, ToolMessage
from langgraph.graph.message import add_messages

@dataclass
class AppContent:
    user_id: str
    tier: str = "standard"

class SavePreferenceInput(BaseModel):
    category: Literal["music", "food", "movie"] = Field(
        description="Category of preference to save"
    )
    preference: str = Field(description="The User's preference value")

@tool(args_schema=SavePreferenceInput)
def save_preference(category: str, preference: str, runtime: ToolRuntime):
    """Save a user's preference to long-term memory"""
    user_id = runtime.context.user_id

    existing = runtime.store.get(("preference", user_id), "data")
    prefs = existing.value if existing else {}

    prefs[category] = preference
    runtime.store.put(["preference", user_id], "data", prefs)

    return Command(
        update = {
            "messages": [
                ToolMessage(
                    content=f"{category} 선호도 '{preference}'를 저장했어요.",
                    tool_call_id=runtime.tool_call_id
                )
            ]
        }
    )




## 실습 (TODO)

지금까지 배운 내용으로 나만의 도구를 만들어봐요!

In [2]:
import time
import datetime
from typing import Literal, Optional
from dataclasses import dataclass
from pydantic import BaseModel, Field
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 1. 컨텍스트 및 입력 스키마 정의
@dataclass
class UserContext:
    user_id: str

class WeatherInput(BaseModel):
    location: str = Field(description="날씨를 조회할 도시 이름 (예: Seoul, Tokyo 등)")
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="온도 단위: 'celsius' 또는 'fahrenheit'"
    )

class PreferenceInput(BaseModel):
    style: Literal["casual", "formal", "sporty", "vintage"] = Field(
        description="선호하는 패션 스타일"
    )
    cold_sensitivity: Literal["high", "medium", "low"] = Field(
        description="추위에 얼마나 민감한지 여부 ('high': 추위를 많이 탐, 'medium': 보통, 'low': 더위를 많이 탐)"
    )

# 2. 날씨 조회 도구 (Tavily 웹 검색 실시간 조회 및 stream_writer 사용)
@tool(args_schema=WeatherInput)
def get_weather(location: str, unit: str = "celsius", runtime: ToolRuntime = None) -> str:
    """웹 검색을 통해 지정된 도시의 실시간 날씨와 기온 정보를 조회합니다."""
    writer = runtime.stream_writer if runtime else None
    
    # 현재 날짜 구하기 (검색어 정확도 향상을 위해 한국어 날짜 형식 사용)
    now = datetime.datetime.now()
    date_query = f"{now.month}월 {now.day}일"
    current_date_str = f"{now.year}년 {now.month}월 {now.day}일"
    
    if writer:
        writer({"step": "searching", "message": f"Tavily 웹 검색을 통해 {location}의 실시간 날씨 정보({current_date_str})를 조회하는 중..."})
        time.sleep(0.5)
    
    try:
        from langchain_tavily import TavilySearch
        # 실시간 날씨 검색을 위한 도구 초기화 및 검색어 작성 (오늘 날짜와 날씨 키워드 조합)
        search_tool = TavilySearch(max_results=3)
        # 한국 뉴스 및 기상 정보 사이트에서 매칭이 잘 되도록 검색어 최적화
        query = f"{location} 날씨 {date_query} 기온 날씨예보"
        search_results = search_tool.invoke({"query": query})
        
        if writer:
            writer({"step": "done", "message": f"{location} 실시간 날씨 검색 완료!"})
            
        return str(search_results)
        
    except Exception as e:
        # 인터넷 미연결 또는 API 키 설정 누락 시 가상의 날씨 데이터 반환 (Fallback)
        if writer:
            writer({"step": "fallback", "message": f"실시간 검색 실패({e}), 가상 날씨 데이터를 생성합니다."})
            time.sleep(0.5)
        
        loc_lower = location.lower()
        if "seoul" in loc_lower or "서울" in loc_lower:
            temp = 5 if unit == "celsius" else 41
            desc = "맑음 (강한 바람)"
        elif "tokyo" in loc_lower or "도쿄" in loc_lower:
            temp = 18 if unit == "celsius" else 64
            desc = "화창함"
        else:
            temp = 12 if unit == "celsius" else 53
            desc = "흐리고 비"
            
        unit_symbol = "°C" if unit == "celsius" else "°F"
        return f"[Fallback 가상 데이터] {location} 날씨: {desc}, 기온: {temp}{unit_symbol} (검색 오류: {e})"

# 3. 옷차림 선호도 저장 도구 (runtime.store 및 Command 활용)
@tool(args_schema=PreferenceInput)
def save_clothing_preference(style: str, cold_sensitivity: str, runtime: ToolRuntime) -> Command:
    """사용자의 옷차림 스타일 선호도와 추위 민감도를 장기 메모리(Store)에 저장합니다."""
    user_id = runtime.context.user_id
    
    pref_data = {
        "style": style,
        "cold_sensitivity": cold_sensitivity
    }
    
    runtime.store.put(("clothing_pref", user_id), "preference", pref_data)
    
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"사용자 '{user_id}'님의 선호 스타일({style})과 추위 민감도({cold_sensitivity})를 장기 메모리에 저장했습니다.",
                    tool_call_id=runtime.tool_call_id
                )
            ]
        }
    )

# 4. 저장된 선호도 조회 도구
@tool
def get_clothing_preference(runtime: ToolRuntime) -> str:
    """장기 메모리(Store)에서 현재 사용자의 옷차림 선호도 데이터를 조회합니다."""
    user_id = runtime.context.user_id
    
    item = runtime.store.get(("clothing_pref", user_id), "preference")
    if item and item.value:
        pref = item.value
        return f"저장된 취향 - 스타일: {pref.get('style')}, 추위 민감도: {pref.get('cold_sensitivity')}"
    else:
        return "저장된 옷차림 취향이 없습니다. 기본 설정(style: casual, cold_sensitivity: medium)을 적용하세요."

# 5. 에이전트 생성
store = InMemoryStore()
checkpointer = InMemorySaver()
model = init_chat_model("openai:gpt-4o-mini")

now = datetime.datetime.now()
current_date_str = f"{now.year}년 {now.month}월 {now.day}일"

agent = create_agent(
    model,
    tools=[get_weather, save_clothing_preference, get_clothing_preference],
    context_schema=UserContext,
    store=store,
    checkpointer=checkpointer,
    system_prompt=(
        "너는 날씨와 사용자의 개인 선호도에 맞춰 옷차림을 추천해주는 스마트 스타일리스트 에이전트야. "
        f"오늘 날짜는 {current_date_str} 이며, 검색어 결과에서 반드시 이 날짜에 해당하는 최신 정보(과거 몇 년 전 기록이나 엉뚱한 날짜가 아님)를 추출해야 해. "
        "응답은 반드시 한국어로 작성해줘.\n"
        "사용자가 날씨나 옷차림 추천을 물어보면:\n"
        "1. 먼저 `get_clothing_preference`를 사용해서 사용자의 저장된 선호도가 있는지 조회해.\n"
        "2. 그 다음 `get_weather`를 통해 해당 지역의 실시간 날씨 정보를 검색해.\n"
        "3. 만약 사용자가 본인의 스타일이나 선호도를 말하면 `save_clothing_preference`로 저장해줘.\n"
        "4. 조회된 날씨 정보에서 현재 기온과 상태를 추출하고, 사용자의 스타일 취향, 그리고 추위 민감도를 종합해서 구체적인 상의, 하의, 외투, 액세서리(우산, 머플러 등)를 추천해줘."
    )
)

# 6. 실행 및 테스트
config = {"configurable": {"thread_id": "user-styling-session-1"}}
user_context = UserContext(user_id="user_elixir")

# 6-1. 사용자 선호도 저장 요청
print("=== 1단계: 선호도 저장 테스트 ===")
inputs_save = {"messages": [{"role": "user", "content": "내 옷차림 선호 스타일은 casual이고, 추위를 많이 타니까(high) 내 취향으로 등록해줘."}]}
for chunk in agent.stream(inputs_save, context=user_context, config=config, stream_mode="custom"):
    print(f"[진행 스트림] {chunk}")

# 상태 확인 및 최종 답변 출력
state = agent.get_state(config)
print(f"\n최종 답변: {state.values['messages'][-1].content}\n")

# 6-2. 날씨 및 옷차림 추천 요청
print("=== 2단계: 날씨 기반 맞춤 옷차림 추천 테스트 ===")
inputs_recommend = {"messages": [{"role": "user", "content": "오늘 울산 날씨는 어때? 나한테 어울리는 옷차림 추천해줘."}]}
for chunk in agent.stream(inputs_recommend, context=user_context, config=config, stream_mode="custom"):
    print(f"[진행 스트림] {chunk}")

state = agent.get_state(config)
print(f"\n최종 답변: {state.values['messages'][-1].content}")


=== 1단계: 선호도 저장 테스트 ===
[진행 스트림] {'step': 'searching', 'message': 'Tavily 웹 검색을 통해 Seoul의 실시간 날씨 정보(2026년 6월 1일)를 조회하는 중...'}
[진행 스트림] {'step': 'done', 'message': 'Seoul 실시간 날씨 검색 완료!'}

최종 답변: 현재 서울의 날씨는 6월 1일로, 기온은 24.4도이며 낮에는 약 29도까지 오를 것으로 예상됩니다. 오늘은 한여름처럼 더운 날씨이므로 자외선과 오존 농도에도 주의해야 합니다. 

사용자의 스타일과 추위 민감도를 고려할 때, 다음과 같은 옷차림을 추천합니다:

### 상의
- **가벼운 반팔 티셔츠 또는 블라우스**: 통기성이 좋고 시원한 소재를 선택하세요.

### 하의
- **면 반바지 또는 가벼운 린넨 팬츠**: 더운 날씨에 맞게 시원하게 입을 수 있는 편안한 하의를 선택하세요.

### 외투
- **얇은 카디건이나 가벼운 아우터**: 아침이나 저녁 기온이 다소 낮을 수 있으니 준비하는 것이 좋습니다.

### 액세서리
- **모자와 선글라스**: 강한 자외선에 대비하기 위해 보호장비로 사용하세요.
- **얇은 머플러**: 자외선으로부터 피부를 보호하는 용도로 사용할 수 있습니다.
- **우산 또는 차양**: 비가 올 가능성이 있으니 필요할 수 있습니다.

### 추가 팁
- 오늘처럼 더운 날씨에는 **수분 보충**을 잊지 마세요. 물이나 이온음료를 챙기는 것이 좋습니다. 

이렇게 준비하시면 오늘 하루를 쾌적하게 보낼 수 있을 것입니다!

=== 2단계: 날씨 기반 맞춤 옷차림 추천 테스트 ===
[진행 스트림] {'step': 'searching', 'message': 'Tavily 웹 검색을 통해 Ulsan의 실시간 날씨 정보(2026년 6월 1일)를 조회하는 중...'}
[진행 스트림] {'step': 'done', 'message': 'Ulsan 실시간 날씨 검색 완료!'}

최종 답변: 현재 울산의 날씨

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **역할 분리**: Part 02는 도구 루프 기초, 이 노트북은 V1 도구의 런타임 확장을 다뤄요
- **`@tool` 데코레이터**: 함수 이름, docstring, 타입 힌트가 도구 메타데이터로 자동 추출돼요
- **`args_schema` + Pydantic**: `Literal` 타입으로 허용값을 제한하고, `Field(description=...)` 으로 파라미터별 설명을 추가해요
- **`ToolRuntime`**: 도구 시그니처에 추가하면 모델에 노출되지 않고 자동 주입돼요. state, context, store, stream_writer에 접근해요
- **`Command` 반환**: 도구 내에서 에이전트 상태를 직접 업데이트해요. `ToolMessage`와 `tool_call_id`를 반드시 포함해야 해요
- **`RemoveMessage`**: 대화 히스토리에서 특정 메시지를 삭제해요. `Command.update`에 포함해서 사용해요
- **`stream_writer`**: 도구 실행 중 진행 상황을 실시간으로 스트리밍해요. `stream_mode="custom"`으로 수신해요


## 다음 노트북 예고

다음 `03-Structured-Output.ipynb`에서는 **구조화된 출력(Structured Output)**을 배워요. 모델이 자유 형식 텍스트 대신 정해진 스키마로 응답을 반환하게 하는 `ToolStrategy`와 `ProviderStrategy` 두 가지 방법을 비교하고, `Content Blocks`를 활용한 복합 출력 처리도 다뤄요.

In [3]:
from typing import Annotated, List, Literal
from dataclasses import dataclass
from pydantic import BaseModel, Field
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, ToolMessage
@dataclass
class AppContext:
    user_id: str
    tier: str = "standard"
class SavePreferencesInput(BaseModel):
    category: Literal["style", "color"] = Field(
        description="저장할 선호도 카테고리 (style: 패션 스타일, color: 선호 색상)"
    )
    preference: str = Field(description="유저가 선호하는 스타일 값 (예: 댄디, 캐주얼, 스트릿)")
@tool(args_schema=SavePreferencesInput)
def save_preference(category: str, preference: str, runtime: ToolRuntime):
    """사용자의 옷 스타일이나 색상 선호도를 메모리에 저장합니다."""
    user_id = runtime.context.user_id
    existing = runtime.store.get(("clothing_preference", user_id), "data")
    prefs = existing.value if existing else {}
    prefs[category] = preference
    runtime.store.put(("clothing_preference", user_id), "data", prefs)
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"[시스템] {category} 선호도에 '{preference}' 스타일이 저장되었습니다.",
                    tool_call_id=runtime.tool_call_id
                )
            ]
        }
    )
class GetWeatherInput(BaseModel):
    location: str = Field(description="날씨 분석을 요청할 대한민국 도시/지역 이름 (예: 서울, 대전, 대구, 부산, 제주, 강릉, 전주 등 전국 모든 지역)")
@tool(args_schema=GetWeatherInput)
def get_current_weather(location: str, runtime: ToolRuntime) -> str:
    """유저가 요청한 대한민국 전역(도시/도 단위)의 날씨 정보를 파악하기 위한 트리거 도구입니다."""
    return f"[날씨 시스템] 사용자가 요청한 지역은 '{location}'입니다. 모델이 가진 지식을 활용해 현재 해당 지역의 대략적인 기온(°C)과 날씨 상황을 설정하고 옷차림을 추천하세요."
recommendation_store = InMemoryStore()
model = init_chat_model("openai:gpt-4o")
fashion_agent = create_agent(
    model,
    tools=[save_preference, get_current_weather],
    store=recommendation_store,
    context_schema=AppContext,
    system_prompt=(
        "You are a professional fashion coordinator. Respond in Korean.\n"
        "당신의 역할은 대한민국 전국 지역의 날씨와 기온을 지능적으로 파악하고, 유저의 선호 스타일과 매칭하여 구체적인 옷차림을 추천하는 것입니다.\n\n"
        "1. 사용자가 본인의 패션 취향을 이야기하면 `save_preference` 도구를 사용해 장기 기억장치에 저장하세요.\n"
        "2. 사용자가 대한민국 내 특정 지역(예: 서울, 대전, 대구, 부산, 인천, 광주, 울산, 제주, 강릉 등 전국 어디나)의 옷차림을 물어보면, 무조건 `get_current_weather` 도구를 먼저 호출하여 지역 검증을 거치세요.\n"
        "3. 도구 호출 이후, 모델(당신)이 가진 최신 배경지식을 총동원하여 해당 지역의 현재 계절과 대략적인 기온(예: '현재 서울은 대략 18°C로 선선한 날씨입니다')을 스스로 설정하여 답변을 시작하세요.\n"
        "4. 설정한 기온과 유저의 선호 스타일(캐주얼, 댄디 등)에 맞춰 아래 가이드라인을 바탕으로 상/하의 및 아우터 조합을 상세하고 센스 있게 제안하세요.\n"
        "   - 10°C ~ 16°C: 가벼운 자켓, 트렌치코트, 셔츠 위에 니트 레이어드, 청바지, 슬랙스\n"
        "   - 17°C ~ 19°C: 맨투맨, 후드티, 얇은 가디건이나 셔츠 겉옷, 면바지\n"
        "   - 20°C ~ 23°C: 얇은 긴팔 티셔츠, 셔츠 단독 착용, 슬랙스\n"
        "   - 24°C 이상: 반팔 티셔츠, 린넨 셔츠, 반바지\n"
        "5. 응답 시 반드시 '현재 [지역명]의 날씨는 X도 안팎으로~' 와 같이 명시하며 친절하게 답변하세요."
    ),
)
premium_ctx = AppContext(user_id="user001", tier = "premium")
r1 = fashion_agent.invoke(
    {"messages": [{"role": "user", "content": "나는 평소에 깔끔한 댄디 룩 스타일을 좋아해"}]},
    context=premium_ctx,
)
print(r1["messages"][-1].content)
saved = recommendation_store.get(("clothing_preference", "user001"), "data")
print("저장된 선호도:", saved.value if saved else "없음")
print("\n--- [추천 요청 테스트] ---")
r2 = fashion_agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨에 맞는 캐주얼 옷차림 추천해줘"}]},
    context=premium_ctx,
)
print(r2["messages"][-1].content)

댄디 스타일을 선호하시는군요! 앞으로 옷차림을 추천할 때 참고하겠습니다. 이어서 특정 지역의 옷차림에 대해 궁금한 점이 있으시면 말씀해 주세요. 전국 어디든지 날씨와 맞춰 적합한 패션을 제안드리겠습니다.
저장된 선호도: {'style': '댄디'}

--- [추천 요청 테스트] ---
현재 서울의 날씨는 대략 18°C로, 선선한 날씨입니다. 캐주얼한 스타일로는 다음과 같은 옷차림을 추천드립니다.

- 상의: 산뜻한 컬러의 맨투맨이나 후드티
- 아우터: 얇은 가디건이나 셔츠를 입으면 좋습니다.
- 하의: 면바지나 청바지를 함께 매치하시면 스타일리시합니다.

이런 조합은 가을의 분위기와 잘 어울리며, 활동하기에도 편안한 스타일입니다.
